# Extracción y Almacenamiento (Chunker)
En este notebook vamos a extraer los documentos para el levantamiento de requerimientos y guardarlos en pgvector para usarlos después como contexto.

In [1]:
!source env/bin/activate
%pip install pypdf langchain langchain-community langchain-google-genai psycopg2-binary pgvector tiktoken dotenv pdfplumber

Note: you may need to restart the kernel to use updated packages.


## Configurar variables de entorno

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

db_name = os.getenv("POSTGRES_DB")
db_password = os.getenv("POSTGRES_PASSWORD")
db_user = os.getenv("POSTGRES_USER")
db_host = os.getenv("POSTGRES_HOST", "localhost")
db_port = os.getenv("POSTGRES_PORT", "5432")

# La URL de conexión al contenedor docker que creamos
CONNECTION_STRING = f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"


## Cargar documento PDF (Ej. ISO/IEC/IEEE 29148)

In [3]:
from langchain_community.document_loaders import PyPDFLoader, PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

pdf_path = "./docs/iso_29148.pdf"

loader = PDFPlumberLoader(
    pdf_path
)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

print(f"Total de chunks creados: {len(chunks)}")
print("Ejemplo de un chunk:", chunks[0].page_content)

Total de chunks creados: 405
Ejemplo de un chunk: Downloaded from https://iranpaper.ir
https://www.tarjomano.com https://www.tarjomano.com
ISO/IEC/
IEEE
INTERNATIONAL 29148
STANDARD
Second edition
2018-11
Systems and software engineering —
Life cycle processes — Requirements
engineering
Ingénierie des systèmes et du logiciel — Processus du cycle de vie —
Ingénierie des exigences
Reference number
ISO/IEC/IEEE 29148:2018(E)
Authorized licensed use limited to: UCLA Library. Downloaded on September 21,2024 at 18:36:23 UTC from IEEE Xplore. Restriction©s apply.
ISO/IEC 2018
©
IEEE 2018


## Guardar en la base de datos (pgvector)

In [12]:
from google import genai

gemini_api_key = os.getenv("GOOGLE_API_KEY", "key not found")

client = genai.Client(
    api_key = gemini_api_key,
    http_options = {
        "timeout": 1000.0
    }
)

for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2

In [14]:
from langchain_community.vectorstores.pgvector import PGVector
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# modelo para los embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model = "models/gemini-embedding-001"
)

# crear la colección y guardar embedings en la db
db = PGVector(
    connection_string=CONNECTION_STRING,
    embedding_function=embeddings,
    collection_name="iso_standards",
    pre_delete_collection=True,
)


/tmp/ipykernel_18193/1882231050.py:10: LangChainPendingDeprecationWarning: This class is pending deprecation and may be removed in a future version. You can swap to using the `PGVector` implementation in `langchain_postgres`. Please read the guidelines in the doc-string of this class to follow prior to migrating as there are some differences between the implementations. See <https://github.com/langchain-ai/langchain-postgres> for details about the new implementation.
  db = PGVector(
/tmp/ipykernel_18193/1882231050.py:10: LangChainPendingDeprecationWarning: Please use JSONB instead of JSON for metadata. This change will allow for more efficient querying that involves filtering based on metadata. Please note that filtering operators have been changed when using JSONB metadata to be prefixed with a $ sign to avoid name collisions with columns. If you're using an existing database, you will need to create a db migration for your metadata column to be JSONB and update your queries to use t

In [15]:
import time

BATCH_SIZE = 50
DELAY = 45  # según tu error (≈43s)

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i+BATCH_SIZE]
    
    print(f"Batch {i} - {i+len(batch)}")
    
    try:
        db.add_documents(batch)
    except Exception as e:
        print(f"Error en batch {i}: {e}")
        time.sleep(DELAY)
        db.add_documents(batch)  # retry simple
    
    if i + BATCH_SIZE < len(chunks):
        time.sleep(DELAY)
 
print("¡Se han guardado todos los chunks exitosamente en pgvector!")


Batch 0 - 50
Batch 50 - 100
Batch 100 - 150
Batch 150 - 200
Batch 200 - 250
Batch 250 - 300
Batch 300 - 350
Batch 350 - 400
Batch 400 - 405
Error en batch 400: Error embedding content: [Errno -3] Temporary failure in name resolution


GoogleGenerativeAIError: Error embedding content: [Errno -3] Temporary failure in name resolution